In [150]:
import pandas as pd
import requests
from io import StringIO

headers= {
    "User-Agent": "Mozilla/5.0(Windows NT 10.0; Win 64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
url = "https://en.wikipedia.org/wiki/List_of_footballers_with_500_or_more_goals"
res = requests.get(url,headers=headers).text
res = StringIO(res)


df=pd.read_html(res)
df

[    0                                                  1
 0 NaN  This section needs additional citations for ve...,
    Rank              Player    Club                  Country and other  \
    Rank              Player  League  Cup Continental Country and other   
 0     1   Cristiano Ronaldo  590[a]   57         172               143   
 1     2       Lionel Messi*  553[b]   71         157               115   
 2     3               Pelé*  604[c]   49          26                83   
 3     4             Romário  545[d]   93          54                64   
 4     5       Ferenc Puskás  516[e]   69          56                84   
 5     6        Josef Bican*  515[f]  137          38                32   
 6     7  Robert Lewandowski  423[g]   62         117                88   
 7     8        Jimmy Jones*  330[h]  286          14                 9   
 8     9        Gerd Müller*  405[i]   92          69                68   
 9    10       Joe Bambrick*  347[j]  253           5     

In [151]:
df= df[3]

In [152]:
df.to_csv("./data/top500goals.csv", index=False)

In [153]:
df=pd.read_csv("./data/top500goals.csv")
df.head()

,Rank,Player,Goals,Matches,Ratio,Career span
0,1,Erwin Helmchen,989+,582,1.70,1924–1951
1,2,Cristiano Ronaldo,975,1337,0.73,2002–present
2,3,Josef Bican,950+,624,1.52,1930–1957
3,4,Ronnie Rooke,934+,1030,0.91,1929–1961
4,5,Lionel Messi,925,1194,0.77,2003–present


In [154]:
import random, math

df["Career span"]= df["Career span"].str.split("–").str[0]


In [155]:
df["Career span"].isna().sum()

np.int64(0)

In [156]:
df.loc[~df['Goals'].str.isnumeric()]
df["Goals"]= [item.replace("+","")for item in df["Goals"]]
df["Goals"]

0     989
1     975
2     950
3     934
4     925
     ... 
78    504
79    504
80    503
81    502
82    500
Name: Goals, Length: 83, dtype: object

In [157]:
df["Goals"] = df["Goals"].astype(int)
df["Career span"] = df["Career span"].astype(int)
df.head()

,Rank,Player,Goals,Matches,Ratio,Career span
0,1,Erwin Helmchen,989,582,1.70,1924
1,2,Cristiano Ronaldo,975,1337,0.73,2002
2,3,Josef Bican,950,624,1.52,1930
3,4,Ronnie Rooke,934,1030,0.91,1929
4,5,Lionel Messi,925,1194,0.77,2003


In [158]:
df["Matches"] = df["Matches"].str.replace("+","") .astype(int)
df.describe()

,Rank,Goals,Matches,Ratio,Career span
count,83.000000,83.000000,83.000000,83.000000,83.00000
mean,42.000000,606.361446,724.915663,0.886024,1945.53012
std,24.103942,122.085394,193.411996,0.267563,26.39306
min,1.000000,500.000000,324.000000,0.530000,1891.00000
25%,21.500000,521.000000,590.500000,0.695000,1927.50000
50%,42.000000,564.000000,711.000000,0.860000,1938.00000
75%,62.500000,631.000000,836.500000,0.960000,1961.00000
max,83.000000,989.000000,1337.000000,1.700000,2010.00000


In [159]:
from datetime import datetime, timedelta
rows =[]
columns = ["name", "date","goals", "assist"]

for row in df.itertuples():
    name=row.Player
    start_date=f"{row._6}-08-01"
    start_date=datetime.strptime(start_date,"%Y-%m-%d")
    goal_ratio= row.Ratio
    goal_sd= random.random()
    goal_sd=goal_sd if goal_sd < 0.7 else 0.7
    assist_sd = 1 - goal_sd
    assist_ratio = 1- goal_ratio
    for i in range(row.Matches):
        name = name
        match_date = timedelta(days=3*i)+ start_date
        goals= random.normalvariate(goal_ratio, goal_sd)
        assists = random.normalvariate(assist_ratio,assist_sd)
        goals =round(goals) if goals > 0 else 0
        assists = round(assists) if assists > 0 else 0
        row = [name, match_date, goals, assists]
        rows.append(row)
games_df=pd.DataFrame(data=rows, columns=columns)
games_df.head()


,name,date,goals,assist
0,Erwin Helmchen,1924-08-01,1,0
1,Erwin Helmchen,1924-08-04,1,0
2,Erwin Helmchen,1924-08-07,2,0
3,Erwin Helmchen,1924-08-10,2,0
4,Erwin Helmchen,1924-08-13,1,1


In [160]:
games_df.shape

(60168, 4)

In [162]:
games_df.sample(10)

,name,date,goals,assist
45410,Des Dickson,1969-12-27,1,0
37459,Zico,1976-03-31,0,1
17623,Stan Mortensen,1943-12-03,1,0
41807,Dennis Westcott,1935-08-12,1,0
54520,Raich Carter,1930-09-14,1,0
59390,Delio Onnis,1971-06-27,1,0
5514,Jimmy Jones,1949-09-19,1,0
42568,Joseph Mermans,1941-01-15,1,0
16789,Eusébio,1964-07-08,1,0
30164,Hughie Gallacher,1924-07-20,1,1


In [172]:
fake_df=games_df.groupby("name").agg(
    matches=("name","count"),
    goals=("goals","sum"),
    assist=("assist", "sum"),
    g_ratio=("goals", "mean"),
    a_ratio=("assist", "mean"),
).reset_index().sort_values(by="goals", ascending=False).reset_index(drop=True)
# fake_df["goal_ratio"]=fake_df["goals"]/ fake_df["matches"]
fake_df.head(10)

,name,matches,goals,assist,g_ratio,a_ratio
0,Josef Bican,624,1094,110,1.753205,0.176282
1,Cristiano Ronaldo,1337,1019,287,0.762154,0.214660
2,Erwin Helmchen,582,1014,11,1.742268,0.018900
3,Lionel Messi,1194,956,236,0.800670,0.197655
4,Ronnie Rooke,1030,911,93,0.884466,0.090291
5,Jimmy Kelly,1004,869,647,0.865538,0.644422
6,Abe Lenstra,850,849,315,0.998824,0.370588
7,Romário,1003,841,191,0.838485,0.190429
8,Jimmy Jones,760,837,96,1.101316,0.126316
9,Pelé,851,806,104,0.947121,0.122209


In [ ]:
for i in range(10):
    print(f"{i} df: {df.loc[i,"Player", "Goals"]}|{i} fake: {fake_df.loc[i,["name", "Goals"]]}
    